# Pick Up & Place Action Designators

Demonstrates `PickUpActionDescription` and `PlaceActionDescription`


**Two approaches:**
- **Direct PyCRAM** — explicit object body, arm, grasp description, and target pose
- **LLM Pipeline** — natural language → LangGraph → CRAM plans → `SimulationBridge.execute_batch`

## 1 · Imports

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

from semantic_digital_twin.world import World
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.datastructures.definitions import TorsoState

from pycram.datastructures.dataclasses import Context
from pycram.datastructures.enums import Arms, ApproachDirection, VerticalAlignment
from pycram.datastructures.pose import PoseStamped
from pycram.datastructures.grasp import GraspDescription
from pycram.language import SequentialPlan
from pycram.motion_executor import simulated_robot
from pycram.testing import setup_world
from pycram.robot_plans import (
    ParkArmsActionDescription,
    MoveTorsoActionDescription,
    NavigateActionDescription,
    PickUpActionDescription,
    PlaceActionDescription,
)
from pycram.designators.location_designator import CostmapLocation

from llmr.adapters import SimulationBridge
from llmr.workflows.graphs.ad_graph import run_with_cache

## 2 · World & Context Setup

In [ ]:
world = setup_world()
context = Context(world, robot)

## 3 · SimulationBridge Setup

In [ ]:
bridge = SimulationBridge(world, robot)

## 4 · Visualize in RViz2 (optional)

Skip this section if ROS2 is not available.

In [ ]:
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
import threading
import rclpy

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [ ]:
_tf_publisher  = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)

---
## 5 · Pick Up & Place — Direct PyCRAM

Full pre-manipulation sequence:
1. Park arms (safe start pose)
2. Raise torso (reach countertop height)
3. Navigate to a base pose near the countertop
4. Pick up the milk carton
5. Place it at a new target pose

In [ ]:
from pycram.view_manager import ViewManager

milk_body  = world.get_body_by_name('milk.stl')
milk_pose  = PoseStamped.from_spatial_type(milk_body.global_pose)
place_pose = PoseStamped.from_list([2.4, 1.8, 1.0], [0, 0, 0, 1], world.root)

# Auto-compute a valid grasp description from the robot manipulator and object pose
arm_view    = ViewManager.get_arm_view(Arms.RIGHT, robot)
grasp_descs = GraspDescription.calculate_grasp_descriptions(arm_view.manipulator, milk_pose)
grasp_desc  = grasp_descs[0]

# CostmapLocation finds a base pose from which the arm can actually reach the milk
nav_loc = CostmapLocation(target=milk_pose, reachable_for=robot, reachable_arm=Arms.RIGHT)

with simulated_robot:
    SequentialPlan(
        context,
        ParkArmsActionDescription(Arms.BOTH),
        MoveTorsoActionDescription([TorsoState.HIGH]),
        NavigateActionDescription(nav_loc),        # IK-validated base pose
        PickUpActionDescription(
            object_designator=milk_body,
            arm=[Arms.RIGHT],
            grasp_description=grasp_desc,
        ),
        PlaceActionDescription(
            object_designator=milk_body,
            target_location=[place_pose],
            arm=Arms.RIGHT,
        ),
    ).perform()

print('Pick & Place (direct) done')

---
## 6 · Pick Up & Place — LLM Pipeline

Natural language → LangGraph (`run_with_cache`) → CRAM plan strings → `SimulationBridge.execute_batch`


In [ ]:
# ── Cache toggle ──────────────────────────────────────────────────────────────
# Set USE_CACHE = True  to return a cached result when available (faster).
# Set USE_CACHE = False to always run the full LLM pipeline (fresh inference).
USE_CACHE = False

# instruction = 'Pick up the milk from the kitchen counter and place it on the table.'
instruction = 'Pick up the milk from the kitchen counter'
result      = run_with_cache(instruction, use_cache=USE_CACHE)
cram_plans  = result.get('cram_plan_response', [])

print(f'LLM generated {len(cram_plans)} plan(s) (cache={USE_CACHE}):')
for i, p in enumerate(cram_plans, 1):
    print(f'  Plan {i}: {p}')

In [ ]:
with simulated_robot:
    results = bridge.execute_batch(cram_plans, arm=Arms.RIGHT)

print('\nPick & Place (LLM pipeline) done  results:', results)

---
## 7 · Pick Up & Place — Hardcoded CRAM Plans for testing (different place targets)

Tests `SimulationBridge.execute_batch` with hand-crafted CRAM strings.
The milk is picked up from the `countertop` and placed on one of several
other surfaces in the apartment world:

| Target tag | World body |
|---|---|
| `island_countertop` | `apartment/island_countertop` (kitchen island) |
| `coffee_table` | `apartment/coffee_table` (living room) |
| `table_area_main` | `apartment/table_area_main` (dining table) |

Run the **reset cell** below first, then pick one target and execute.

In [ ]:
# ── Reset milk to countertop before each hardcoded test ───────────────────────
from semantic_digital_twin.spatial_types.spatial_types import HomogeneousTransformationMatrix
world.get_body_by_name('milk.stl').parent_connection.origin = (
    HomogeneousTransformationMatrix.from_xyz_rpy(2.37, 2.0, 1.05,
                                                  reference_frame=world.root)
)
print('Milk reset to countertop pose')

In [ ]:
# ── Hardcoded CRAM plans — choose a place target ──────────────────────────────
#
# Swap the `place_target` value to test different surfaces:
#   'island_countertop'  →  apartment/island_countertop  (kitchen island)
#   'coffee_table'       →  apartment/coffee_table       (living room)
#   'table_area_main'    →  apartment/table_area_main    (dining table)

place_target = 'table_area_main'

hardcoded_plans = [
    (
        "(an action (type PickingUp) "
        "(object (:tag milk (an object (type Substance size medium color white "
        "texture smooth material liquid weight light condition whole)))) "
        "(source (a location "
        f"(on (:tag countertop (an object (type surface material wood condition whole)))))))"
    ),
    (
        "(an action (type Placing) "
        "(object (:tag milk (an object (type Substance size medium color white "
        "texture smooth material liquid weight light condition whole)))) "
        f"(target (a location "
        f"(on (:tag {place_target} (an object (type surface material wood condition whole)))))))"
    ),
]

print(f'Place target: {place_target}')
print(f'Plan 1: {hardcoded_plans[0]}')
print(f'Plan 2: {hardcoded_plans[1]}')

In [ ]:
# ── Execute hardcoded plans ────────────────────────────────────────────────────
milk_body_hc  = world.get_body_by_name('milk.stl')
milk_pose_hc  = PoseStamped.from_spatial_type(milk_body_hc.global_pose)
nav_loc_hc    = CostmapLocation(target=milk_pose_hc, reachable_for=robot, reachable_arm=Arms.RIGHT)

with simulated_robot:
    SequentialPlan(
        context,
        ParkArmsActionDescription(Arms.BOTH),
        MoveTorsoActionDescription([TorsoState.HIGH]),
        NavigateActionDescription(nav_loc_hc),
    ).perform()
    results = bridge.execute_batch(hardcoded_plans, arm=Arms.RIGHT)

print(f'\nPick & Place (hardcoded → {place_target}) done  results:', results)

---
## Shutdown ROS2 Node

In [ ]:
try:
    _ros_node.destroy_node()
    rclpy.shutdown()
    print('ROS2 node stopped')
except Exception:
    print('(ROS2 node was not running)')